# Backend Sandbox

Use this notebook to inspect the archive/search databases and try FTS queries interactively.

In [1]:
from pathlib import Path
import json
import sys

BACKEND_DIR = Path.cwd()
if BACKEND_DIR.name != "backend":
    BACKEND_DIR = BACKEND_DIR / "backend"

PROJECT_ROOT = BACKEND_DIR.parent
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("BACKEND_DIR:", BACKEND_DIR)


PROJECT_ROOT: /Users/rf50/uchicago/maroon/scrape-maroon-archives
BACKEND_DIR: /Users/rf50/uchicago/maroon/scrape-maroon-archives/backend


In [2]:
from db import ARCHIVE_DB, CHUNKS_DB, fetch_chunk, fetch_document, get_archive_connection, get_chunks_connection
from search_fts import search_fts

print("ARCHIVE_DB exists:", ARCHIVE_DB.exists(), ARCHIVE_DB)
print("CHUNKS_DB exists:", CHUNKS_DB.exists(), CHUNKS_DB)


ARCHIVE_DB exists: True /Users/rf50/uchicago/maroon/scrape-maroon-archives/output/metadata/archive.db
CHUNKS_DB exists: True /Users/rf50/uchicago/maroon/scrape-maroon-archives/output/metadata/chunks.db


In [3]:
archive_conn = get_archive_connection()
chunks_conn = get_chunks_connection()

try:
    document_count = archive_conn.execute("select count(*) from documents").fetchone()[0]
    chunk_count = chunks_conn.execute("select count(*) from chunks").fetchone()[0]
    print("documents:", document_count)
    print("chunks:", chunk_count)
finally:
    archive_conn.close()
    chunks_conn.close()


documents: 3611
chunks: 89246


- On average 25 chunks per document

In [10]:
query = "crimes specifically involving bicycles or cyclists"
results = search_fts(query, limit=10)
print("query:", query)
print("num results:", len(results))
for i, row in enumerate(results, start=1):
    print("-" * 100)
    print(i, row["chunk_id"], row["date"], row["score"])
    print(row["text"][:])


query: crimes specifically involving bicycles or cyclists
num results: 0


In [5]:
sample_chunk_id = results[0]["chunk_id"] if results else None
if sample_chunk_id:
    chunk = fetch_chunk(sample_chunk_id)
    doc = fetch_document(chunk["doc_id"])
    print("chunk_id:", chunk["chunk_id"])
    print("doc_id:", chunk["doc_id"])
    print("pdf_path:", doc["pdf_path"])
    print("text_path:", doc["text_path"])
    print("chunk preview:")
    print(chunk["text"][:1200])


chunk_id: mvol-0004-1956-0224::chunk-0083
doc_id: mvol-0004-1956-0224
pdf_path: /Users/rf50/uchicago/maroon/scrape-maroon-archives/output/pdfs/1956/02/mvol-0004-1956-0224.pdf
text_path: /Users/rf50/uchicago/maroon/scrape-maroon-archives/output/plain_text/1956/02/mvol-0004-1956-0224.txt
chunk preview:
Hutchins to persist in his decision to Tesign as Chancellor. The Council urges Mr. Hutchins to see his way clear to changing his decision.”The Council will meet again Thursday afternoon to elect a committee of five to meet withthe Board of Trustees to discuss the selection of a successor to Hutchins, but the Council is also drafting a memorial to Hutchins Urging him to remain. UC to reopen Jan. 2 Dean Robert M. Strozier In• special announcement last night to the University community emphasized that the University will re-open on Jan. 2, regardless of whether or notthe strike has been settled. “We Hope to settle the strike, of course, before then,” he said. wheels roll out shocked comments 

In [6]:
custom_query = "student protests"
custom_results = search_fts(custom_query, limit=10)

summary = [
    {
        "chunk_id": row["chunk_id"],
        "doc_id": row["doc_id"],
        "date": row["date"],
        "score": row["score"],
        "preview": row["text"][:220],
    }
    for row in custom_results
]
print(json.dumps(summary, indent=2))

[
  {
    "chunk_id": "mvol-0004-1930-1030::chunk-0008",
    "doc_id": "mvol-0004-1930-1030",
    "date": "1930-10-30",
    "score": -8.940852445540193,
    "preview": "veatch sophomore ASSISTANTS HERBERT BERMAN JOHN CLANCY RICHARD DEUTSCH NORMAN JORGENSON DAMON FULLER EDGAR GOLDSMITH CHARLES HOWE CHESTER WARD WOMAN EDITORS ALBERTA KILLIE ING RED PETERSEN ELEANOR WILSONET. I/. An ETH mi"
  },
  {
    "chunk_id": "mvol-0004-1962-0206::chunk-0004",
    "doc_id": "mvol-0004-1962-0206",
    "date": "1962-02-06",
    "score": -8.757943528366807,
    "preview": "Accordingly the permission of January 24 th is hereby withdrawn andany further sit-ins or demonstrations ofany kind are prohibited anywhere in the Administration Building or in any other University building.4. Each of yo"
  },
  {
    "chunk_id": "mvol-0004-1962-0420::chunk-0013",
    "doc_id": "mvol-0004-1962-0420",
    "date": "1962-04-20",
    "score": -8.5687587395928,
    "preview": "It is reprinted are, no ones conscience can b

In [ ]:
from search_vector import search_vector
from search import search

vector_query = "student protests on campus"
vector_results = search_vector(vector_query, limit=5, backend="sentence-transformers")
print("vector query:", vector_query)
print("num vector results:", len(vector_results))
for i, row in enumerate(vector_results, start=1):
    print("-" * 100)
    print(i, row["chunk_id"], row["date"], row["score"], row.get("embedding_backend"))
    print(row["text"][:800])


In [ ]:
hybrid_query = "student protests on campus"
hybrid_results = search(hybrid_query, limit=5, backend="sentence-transformers")
print("query:", hybrid_results["query"])
print("vector backend:", hybrid_results["vector_backend"])
print("chunk results:", len(hybrid_results["chunk_results"]))
print("document results:", len(hybrid_results["document_results"]))

for i, row in enumerate(hybrid_results["chunk_results"], start=1):
    print("-" * 100)
    print(i, row["chunk_id"], row["date"], row["combined_score"])
    print(row["snippet"])

print("\nTop documents")
for i, row in enumerate(hybrid_results["document_results"], start=1):
    print("-" * 100)
    print(i, row["doc_id"], row["date"], row["doc_score"], row["pdf_path"])
    print("best chunk:", row["best_chunk_id"], "chunks in doc result:", row["chunk_count"])
